# အခန်း ၀၁ - AI အေးဂျင့်များအကြောင်း မိတ်ဆက်ခြင်း

**AI အေးဂျင့်စတင်သူများအတွက်** သင်တန်း၏ ပထမဆုံး အခန်းသို့ ကြိုဆိုပါသည်!

**AI အေးဂျင့်** ဆိုသည်မှာ အကြီးစားဘာသာစကားမော်ဒယ် (LLM) ကို အတွေးအခေါ်အင်ဂျင်အဖြစ် အသုံးပြု၍ အသုံးပြုသူအတွက် ရည်ရွယ်ချက်တစ်ခုကို ဆောင်ရွက်ပေးနိုင်သော *လုပ်ဆောင်ချက်များ* (actions) ကို အလုံးစုံကမ္ဘာမှာ လုပ်ဆောင်နိုင်သော ပရိုဂရမ်တစ်ခုဖြစ်သည် — API ခေါ်ရန်၊ ဒေတာဘေ့စ်များ ဆွဲ႐ွေးရန်၊ သို့မဟုတ် ကုဒ်များ ပြေးဆွဲရန် အပါအဝင်။

ဤ notebook တွင် သင်၏ ပထမဆုံး အေးဂျင့်ဖြစ်သော **ခရီးသွား အေးဂျင့်** ကို တည်ဆောက်ပါမည်။ ဤအေးဂျင့်သည် ဖိတ်ကြားရာနေရာများကို အကြံပြုပေးနိုင်သည်။ အတူတကွ သင်သည် -

၁။ **Microsoft Agent Framework** ကို အသုံးပြုကာ Microsoft Foundry Agent Service နှင့် ချိတ်ဆက်နည်း။
၂။ အေးဂျင့်အား **ကိရိယာ** တစ်ခုပေးခြင်း — ၎င်းကို ခေါ်နိုင်သော ရိုးရိုး Python function တစ်ခု။
၃။ အေးဂျင့်ကို ပြေးစေပြီး ၎င်း၏ တုံ့ပြန်မှုကို စစ်ဆေးခြင်း။
၄။ အေးဂျင့်၏ တုံ့ပြန်ချက်ကို တစ်ချို့ချို့ Token အလိုက် ဆက်တိုက်လွှတ်ခြင်း။


## တပ်ဆင်မှု

ဤ notebook ကို အလုပ်လုပ်မပြေးမီ၊ သင့်တွင်ရှိရမည့်အချက်များမှာ-

1. **Microsoft Foundry စီမံကိန်းတစ်ခု** အတွင်း chat မော်ဒယ်တင်ပြီးဖြစ်ပါသည် (ဥပမာ - `gpt-5-mini`)။
2. **Azure CLI ဖြင့် လော့ဂ်အင်ဝင်ပြီးဖြစ်ပါသည်** — သင့် terminal တွင် `az login` ဖြင့် ဖြေရှင်းပါ။
3. **လိုအပ်သော ပတ်ဝန်းကျင် environment variables များ စီစဉ်ထားပါ။**
   - `AZURE_AI_PROJECT_ENDPOINT` — သင့် Microsoft Foundry စီမံကိန်း endpoint ဖြစ်သည်။
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — တင်ထားသော မော်ဒယ်အမည်ဖြစ်သည်။

အောက်က cell သည် သင့်လို Python packages များကို တပ်ဆင်ပေးပါမည်။


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from agent_framework import tool

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## သင်၏ ပထမဆုံး Agent ကို ဖန်တီးခြင်း

Agent တစ်ခုအတွက် လိုအပ်တာနှစ်ခုရှိသည်။

- **ညွှန်ကြားချက်များ** သည် ၎င်းအကြောင်း *သူ* ဟု၊ နှင့် *ဘယ်လိုသွင်ပြင်မှုပေးရမည်* ဆိုတာ (စနစ် prompt) ပြောပြသည်။
- **ကိရိယာများ** — agent သည် အသုံးပြုနိုင်သော Python function များဖြစ်ပြီး `@tool` ဖြင့် အမှတ်အသားပြုထားသည်။ ၎င်းကို အချက်အလက် ရယူရန် သို့မဟုတ် လုပ်ဆောင်ချက်များ ပြုလုပ်ရန် ခေါ်နိုင်သည်။

အောက်တွင် လူကြိုက်များသော ခရီးသွားစရိတ်များ စာရင်းပြန်ပေးသည့် ကိရိယာ စမတင်ထားသည်။ အသုံးပြုသူသည် ခရီးသွားအကြံပေးမှု ရှာမေးသည်အခါ agent သည် ဤကိရိယာကို အသုံးပြုမည်ဖြစ်သည်။


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = provider.as_agent(
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
    tools=[get_destinations],
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## တုံ့ပြန်ချက်များ ကို စတင်ထုတ်လွှင့်ခြင်း

ပိုမိုလှုပ်ရှားမှုမြင့်မားသော အတွေ့အကြုံအတွက် သင်သည် ကိုယ်စားလှယ်၏ တုံ့ပြန်ချက်ကို **စတင်ထုတ်လွှင့်** နိုင်သည်။ ပြည့်စုံသော အဖြေကို စောင့်ဆိုင်းရန် အစား ကိုယ်စားလှယ်သည် ဖန်တီးပေးသည့် စာသားအပိုင်းများကို တိတိကျကျ ထုတ်လွှင့်ပေးသည်။ ၎င်းသည် သင်သည် အချိန်နှင့်အမျှ အထွက်များကို ပြသလိုသော စကားပြော မျက်နှာပြင်များတွင် အထူးအသုံးဝင်သည်။


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## အကျဉ်းချုပ်

ဒီသင်ခန်းစာမှာ သင်တတ်ခဲ့တာတွေကတော့-

- **Microsoft Foundry Agent Service** နဲ့ ချိတ်ဆက်ဖို့ `FoundryChatClient` ဖြင့် provider တစ်ခု ဖန်တီးနည်း
- **`@tool` decorator ကို အသုံးပြုပြီး tool တစ်ခု ကို သတ်မှတ်ပေးခြင်း၊ agent က သင့် Python function များကို ခေါ်နိုင်ရန်။**
- **အသုံးပြုသူ စာတိုက်နှင့် Agent ကို run ပြီး ၎င်း၏ တုံ့ပြန်ချက်ကို ပုံနှိပ်ပေးခြင်း။**
- **ရလဒ်ကို စက်ရုပ်လက်တွေ့ချိန် real-time output အဖြစ် ပြသရန် Stream ပြုလုပ်ခြင်း။**

နောက်ထပ် သင်ခန်းစာမှာ agentic framework များကို နက်ရှိုင်းစွာ လေ့လာကြည့်ပြီး agent များအား ပိုပြီး အားကောင်းလာစေရန် tool များနှင့် multi-step reasoning ကွက်လပ်များ ပေးပို့ပုံ လေ့လာသွားမှာ ဖြစ်ပါတယ်။


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ပြောကြားချက်**
ဤစာတမ်းကို AI ဘာသာပြန်ဝန်ဆောင်မှု [Co-op Translator](https://github.com/Azure/co-op-translator) အသုံးပြု၍ ဘာသာပြန်ထားပါသည်။ ကျွန်ုပ်တို့သည် တိကျမှန်ကန်မှုအတွက် ကြိုးပမ်းနေသော်လည်း၊ စက်ကိရိယာဘာသာပြန်ခြင်းများတွင် အမှားများ သို့မဟုတ် မှားယွင်းချက်များ ပါဝင်နိုင်ကြောင်း သတိပြုပါရန် လိုအပ်ပါသည်။ မူလစာတမ်းကို မူရင်းဘာသာဖြင့်သာ ယုံကြည်စိတ်ချရသော အချက်အလက်အဖြစ် သတ်မှတ်သင့်သည်။ အရေးကြီးသည့် သတင်းအချက်အလက်များအတွက် ပရော်ဖက်ရှင်နယ် လူသားဘာသာပြန်သူဝန်ဆောင်မှုကို အကြံပြုပါသည်။ ဤဘာသာပြန်ချက်ကို အသုံးပြုခြင်းမှ ဖြစ်ပေါ်လာသော နားလည်မှုကွာခြားမှုများ သို့မဟုတ် မမှန်ကန်သော အသုံးပြုမှုများအတွက် ကျွန်ုပ်တို့ တာဝန်မခံပါ။
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
